In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error, make_scorer

In [16]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Data/top40_cell_cycle.csv")

# Separate features and target
X = df.drop(columns=['phase', 'age'])
y = df['age']

feature_names = X.columns.tolist()
X = X.to_numpy()
y = y.to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=949)

LOCO

In [6]:
import sys
sys.path.append("/Users/mariahloehr/IICD/IICD/feature_importance")

In [7]:
import locomp
from locomp import *
from locomp.MLmodels import *
from locomp.util_locomp import *
import itertools
import importlib
from sklearn.base import BaseEstimator, RegressorMixin, clone
import itertools
from functools import partial
import multiprocessing as mp
import re

In [17]:
# define fit_func
def MLPreg(X,Y,X1):
    mlp = MLPRegressor(max_iter=100,
                       random_state=949,
                       hidden_layer_sizes = (300,300),
                       learning_rate = 'adaptive', 
                       learning_rate_init = 0.001, 
                       alpha = 0.001).fit(X,Y)
    return mlp.predict(X1)

In [18]:
J1 = 0
J2 = 1
m_ratio = 0.2
n_ratio = 0.2
B = 5000
fit_func = MLPreg

In [ ]:
predictions, in_mp_obs, in_mp_feature = predictMPReg(X_train, y_train, X_test, n_ratio, m_ratio, B, fit_func)

# Aggregate predictions from all minipatch models
mean_pred = np.mean(predictions, axis=0)

/Users/mariahloehr/IICD/IICD/sklearn-env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mariahloehr/IICD/IICD/sklearn-env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mariahloehr/IICD/IICD/sklearn-env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mariahloehr/IICD/IICD/sklearn-env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization has

In [11]:
rmse = root_mean_squared_error(y_test, mean_pred)
print("Minipatch Ensemble RMSE:", rmse)

Minipatch Ensemble RMSE: 1.8833455737846017


In [12]:
# For test set
y = df['age']  # (reversing making it a numpy array)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=949)

df_test = pd.DataFrame({
    'true_age': y_test,
    'pred_age': mean_pred,
    'phase': df.loc[y_test.index, 'phase']  # get phase for train samples
})

rmse_per_phase_test = df_test.groupby('phase').apply(
    lambda x: root_mean_squared_error(x['true_age'], x['pred_age'])
)

print("\nRMSE per phase (Test):")
print(rmse_per_phase_test)


RMSE per phase (Test):
phase
G0    1.957166
G1    1.514709
G2    2.431044
M     8.727414
S     1.586754
dtype: float64


/var/folders/1s/bvxr71hj0hqgyk_jk6k7wkm80000gn/T/ipykernel_37662/2290763584.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rmse_per_phase_test = df_test.groupby('phase').apply(


In [13]:
# === Load existing results DataFrame ===
results_df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Bar Plot/minipatch_results.csv", index_col=0)

# === Set values ===
model_name = "MLP MP"  # or whatever is appropriate
results_df.loc[model_name, 'Overall'] = rmse

# Fill in per-phase RMSEs
for phase in ['G0', 'G1', 'G2', 'M', 'S']:
    if phase in rmse_per_phase_test.index:
        results_df.loc[model_name, phase] = rmse_per_phase_test[phase]

# === Save updated results ===
results_df.to_csv("/Users/mariahloehr/IICD/IICD/Bar Plot/minipatch_results.csv")